# Detectarea textelor human / AI in limba romana

Qwen3.5-4B, LoRA si Unsloth. Rezultate exportate din Colab; celulele temporare de diagnostic au fost omise.


## Setup


In [ ]:
from pathlib import Path
import re
OFFLINE_VALIDATION = False  # Numai companionul local de verificare setează True.
RUN_MODE = "train"  # "prepare", "pilot", "train", "calibrate", "evaluate"
RUN_NAME = "qwen35_4b_ro_t4_kv_v1"
MODEL_NAME = "unsloth/Qwen3.5-4B"
MAX_SEQ_LENGTH = 12288
SEED = 3407
TRAIN_BATCH_SIZE = 1
ENABLE_T4_KV_FIX = True
GRADIENT_ACCUMULATION = 16
NUM_TRAIN_EPOCHS = 1
LEARNING_RATE = 2e-4
CONFIRM_FULL_TRAINING = True  # True numai după pilot, în modul train.
RESUME_CHECKPOINT = "/content/drive/MyDrive/SDCGIA/experiments/qwen35_4b_ro_t4_kv_v1/checkpoints/checkpoint-160"  # Calea completă către checkpoint-N din ACELAȘI experiment.
BUNDLE_PATH = Path("/content/drive/MyDrive/SDCGIA/colab/SDCGIA_human_ai_v1.zip")
EXTRACT_ROOT = Path("/content/sdcgia_human_ai_v1")
SAVE_ROOT = Path("/content/drive/MyDrive/SDCGIA/experiments")
EXPECTED_MANIFEST_SHA = '28ce73f01101e43c8285144dacb77921f387412efcd2dd10ad2731b78dc1a8ae'
assert RUN_MODE in {"prepare", "pilot", "train", "calibrate", "evaluate"}
assert re.fullmatch(r"[A-Za-z0-9_-]+", RUN_NAME)
assert MAX_SEQ_LENGTH >= 512
print("Mod:", RUN_MODE, "| Model:", MODEL_NAME, "| Context:", MAX_SEQ_LENGTH)

Mod: train | Model: unsloth/Qwen3.5-4B | Context: 12288


### 2. Google Drive și GPU


In [ ]:
if not OFFLINE_VALIDATION:
    from google.colab import drive
    drive.mount("/content/drive")
    import subprocess
    check = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True)
    if check.returncode:
        raise RuntimeError("GPU NVIDIA indisponibil. Runtime > Change runtime type > GPU.")
    print(check.stdout)
else:
    print("OFFLINE: Drive și verificarea GPU nu sunt executate.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
name, memory.total [MiB]
Tesla T4, 15360 MiB



### 3. Instalare — celula originală


In [ ]:
if not OFFLINE_VALIDATION:
    installation_result = get_ipython().run_cell('%%capture\nimport os, importlib.util\n!pip install --upgrade -qqq uv\nif importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):\n    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"\n    except: _numpy = "numpy"; _pil = "pillow"\n    !uv pip install -qqq \\\n        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \\\n        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \\\n        "unsloth[base] @ git+https://github.com/unslothai/unsloth"\n    !uv pip install -qqq --no-deps "torchcodec==0.7.0"\nelif importlib.util.find_spec("unsloth") is None:\n    !uv pip install -qqq unsloth\n!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo\n!uv pip install transformers==5.2.0\n!unsloth install-kernels\n!uv pip install --no-deps --upgrade "torchao>=0.16.0"')
    if installation_result.error_before_exec or installation_result.error_in_exec:
        raise RuntimeError("Instalarea a eșuat; nu continuați.")
else:
    print("OFFLINE: instalarea nu este executată.")


In [ ]:
# 3.1 Restaurare reproductibila Unsloth pentru checkpoint-160.
# Sursa oficiala este verificata fata de hash-ul experimentului.
# Nu modifica run_config.json, checkpointuri sau date.
import hashlib, importlib, importlib.metadata as restore_metadata
import json, subprocess, sys, urllib.request
from pathlib import Path
RESTORE_UNSLOTH_REVISION = '3594ed54db37538c2a7fe975a3d12ac495c0cb6a'
RESTORE_DECISION_SHA256 = '1b4a9d45965e4cfa93995d634d2104637f2c559595f06b104cc1e94075a45b67'
restore_run = Path('/content/drive/MyDrive/SDCGIA/experiments/qwen35_4b_ro_t4_kv_v1')
assert json.loads((restore_run/'run_config.json').read_text())['decision_source_sha256'] == RESTORE_DECISION_SHA256
restore_path = Path(restore_metadata.distribution('unsloth').locate_file('unsloth/models/decision.py'))
RESTORE_RESTART_REQUIRED = False
if hashlib.sha256(restore_path.read_bytes()).hexdigest() != RESTORE_DECISION_SHA256:
    restore_url = f'https://raw.githubusercontent.com/unslothai/unsloth/{RESTORE_UNSLOTH_REVISION}/unsloth/models/decision.py'
    with urllib.request.urlopen(restore_url, timeout=30) as response:
        verified_source = response.read()
    assert hashlib.sha256(verified_source).hexdigest() == RESTORE_DECISION_SHA256
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '--force-reinstall',
        f'git+https://github.com/unslothai/unsloth.git@{RESTORE_UNSLOTH_REVISION}'],check=True)
    importlib.invalidate_caches()
    RESTORE_RESTART_REQUIRED = True
assert restore_metadata.version('unsloth') == '2026.10.7'
restore_path = Path(restore_metadata.distribution('unsloth').locate_file('unsloth/models/decision.py'))
assert hashlib.sha256(restore_path.read_bytes()).hexdigest() == RESTORE_DECISION_SHA256
print('SURSA ORIGINALA RESTAURATA SI VERIFICATA:', RESTORE_DECISION_SHA256)
print('RESTART NECESAR:', RESTORE_RESTART_REQUIRED)
if RESTORE_RESTART_REQUIRED:
    print('Repornește sesiunea înainte de încărcarea modelului. Nu rerula instalarea originala dupa restart; ruleaza configuratia, Drive si aceasta verificare.')


SURSA ORIGINALA RESTAURATA SI VERIFICATA: 1b4a9d45965e4cfa93995d634d2104637f2c559595f06b104cc1e94075a45b67
RESTART NECESAR: False


## Steps


In [ ]:
import json, hashlib, zipfile, types
with zipfile.ZipFile(BUNDLE_PATH) as archive:
    manifest_bytes = archive.read("bundle_manifest.json")
    assert hashlib.sha256(manifest_bytes).hexdigest() == EXPECTED_MANIFEST_SHA, "Pachet greșit sau modificat"
    bundle_manifest = json.loads(manifest_bytes)
    runtime_bytes = archive.read("colab_human_ai_runtime.py")
    assert hashlib.sha256(runtime_bytes).hexdigest() == bundle_manifest["files"]["colab_human_ai_runtime.py"]
runtime = types.ModuleType("sdcgia_runtime")
exec(compile(runtime_bytes, "colab_human_ai_runtime.py", "exec"), runtime.__dict__)
runtime.unpack_verified(BUNDLE_PATH, EXTRACT_ROOT, EXPECTED_MANIFEST_SHA)
data, metadata = runtime.load_splits(EXTRACT_ROOT)
assert {k: len(v) for k, v in data.items()} == bundle_manifest["counts"]
for split in data:
    from collections import Counter
    print(split, len(data[split]), dict(Counter(m["label"] for m in metadata[split])))
print("Verificări de integritate și separare: OK. Textele nu au fost modificate.")


train 4448 {'ai': 1811, 'human': 2637}
calibration 955 {'ai': 390, 'human': 565}
test 954 {'ai': 389, 'human': 565}
Verificări de integritate și separare: OK. Textele nu au fost modificate.


### 5. Tokenizer și auditul lungimilor — fără antrenare


In [ ]:
if not OFFLINE_VALIDATION:
    import unsloth  # înainte de transformers/torch
    from transformers import AutoTokenizer
    from unsloth.models.clef import encode_record
    tokenizer_audit = AutoTokenizer.from_pretrained(MODEL_NAME)
    def audit_encode(row, budget):
        return encode_record(tokenizer_audit, row, max_length=budget)
    length_report = runtime.inspect_lengths(data, metadata, audit_encode, MAX_SEQ_LENGTH)
    analysis_dir = SAVE_ROOT / RUN_NAME / ("analysis_ctx" + str(MAX_SEQ_LENGTH))
    runtime.save_json(analysis_dir / "token_lengths.json", length_report)
    print(json.dumps({k: v for k, v in length_report.items() if k != "records"}, indent=2))
    CONTEXT_OK = length_report["over_context"] == 0
    print("Context suficient:", CONTEXT_OK)
    if not CONTEXT_OK:
        print("OPRIRE DE SIGURANȚĂ: pilotul/antrenarea sunt blocate, nu se acceptă trunchiere implicită.")
else:
    CONTEXT_OK = False
    print("OFFLINE: tokenizarea reală nu este verificată; nu există rezultat GPU sau de lungime în tokeni.")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


/usr/local/lib/python3.13/dist-packages/unsloth/models/rl.py:1479: UserWarning: Torchaudio's I/O functions now support per-call backend dispatch. Importing backend implementation directly is no longer guaranteed to work. Please use `backend` keyword with load/save/info function, instead of calling the underlying implementation directly.
  if getattr(module, "prepare_peft_model", None) is original:


{
  "max_seq_length": 12288,
  "splits": {
    "train": {
      "rows": 4448,
      "max_tokens": 10465,
      "median_tokens": 566,
      "p95_tokens": 1399,
      "over_context": 0
    },
    "calibration": {
      "rows": 955,
      "max_tokens": 5484,
      "median_tokens": 603,
      "p95_tokens": 1676,
      "over_context": 0
    },
    "test": {
      "rows": 954,
      "max_tokens": 8185,
      "median_tokens": 597,
      "p95_tokens": 2182,
      "over_context": 0
    }
  },
  "required_max_tokens": 10465,
  "over_context": 0
}
Context suficient: True


### 5.1. Corecție atenție T4 — GQA explicit


In [ ]:
"""Narrow, reversible Transformers 5.2 / PyTorch 2.8 T4 GQA workaround.

No imports of torch/transformers at import, no disk changes, no weight changes.
The original HF sdpa_attention_forward performs repeat_kv when this selector
returns False. Its mask/dropout/scale/causal handling stays unchanged.
"""
import hashlib
import inspect
from pathlib import Path

POLICY = 't4_explicit_kv_transformers_selector_v1'
EXPECTED_VERSIONS = {'torch': '2.8.0', 'transformers': '5.2.0',
                     'unsloth': '2026.10.7', 'unsloth_zoo': '2026.10.7'}


class T4ExplicitKVFix:
    def __init__(self, torch_module, sdpa_module, versions):
        for name, expected in EXPECTED_VERSIONS.items():
            actual = versions.get(name, '').split('+')[0]
            if actual != expected:
                raise RuntimeError(f'Runtime nevalidat pentru corecție: {name}={actual}; așteptat {expected}.')
        if tuple(torch_module.cuda.get_device_capability(0)) != (7, 5):
            raise RuntimeError('Corecția este restrânsă la GPU Turing (compute capability 7.5).')
        original = getattr(sdpa_module, 'use_gqa_in_sdpa', None)
        if not callable(original) or not callable(getattr(sdpa_module, 'repeat_kv', None)):
            raise RuntimeError('API Transformers diferit; corecția nu a fost aplicată.')
        if hasattr(original, '_sdcgia_fix'):
            raise RuntimeError('Corecția este deja aplicată. Repornește sesiunea înainte de rerulare.')
        if list(inspect.signature(original).parameters) != ['attention_mask', 'key']:
            raise RuntimeError('Semnătura selectorului GQA diferă de cea verificată.')
        self.torch, self.module, self.original = torch_module, sdpa_module, original
        self.calls, self.shapes = 0, set()
        self.versions = {k: versions[k] for k in EXPECTED_VERSIONS}
        try:
            source = Path(inspect.getsourcefile(original)).read_bytes()
            self.source_sha256 = hashlib.sha256(source).hexdigest()
        except (OSError, TypeError):
            self.source_sha256 = None

        def selector(attention_mask, key):
            targeted = (attention_mask is None and key.device.type == 'cuda'
                        and key.ndim == 4 and key.shape[1] == 4 and key.shape[-1] == 256
                        and key.dtype == self.torch.float16
                        and tuple(self.torch.cuda.get_device_capability(key.device)) == (7, 5))
            if targeted:
                self.calls += 1
                if len(self.shapes) < 32:
                    self.shapes.add(tuple(int(n) for n in key.shape))
                return False
            return self.original(attention_mask, key)

        selector._sdcgia_fix = self
        self.selector = selector
        self.module.use_gqa_in_sdpa = selector

    def assert_installed(self):
        if self.module.use_gqa_in_sdpa is not self.selector:
            raise RuntimeError('Selectorul de atenție a fost înlocuit ulterior; nu continua pilotul.')

    def reset_counters(self):
        self.calls, self.shapes = 0, set()

    def config(self):
        return {'policy': POLICY, 'scope': 'T4 CUDA float16 K/V heads=4 head_dim=256 mask=None',
                'versions': self.versions, 'transformers_sdpa_source_sha256': self.source_sha256,
                'model_weights_changed': False, 'dtype_changed': False, 'article_text_changed': False,
                'selector': 'explicit_repeat_kv_via_existing_transformers_path'}

    def snapshot(self):
        return {**self.config(), 'bypassed_native_gqa_calls': self.calls,
                'observed_key_shapes_before_repeat': [list(s) for s in sorted(self.shapes)]}

    def restore(self):
        self.assert_installed()
        self.module.use_gqa_in_sdpa = self.original


def inspect_attention_events(events):
    """Inspect executed ATen operators, not merely available/enabled backend flags."""
    operators = {event.key: int(event.count) for event in events
                 if 'scaled_dot_product' in event.key.lower()}
    forward = operators.get('aten::_scaled_dot_product_efficient_attention', 0)
    backward = operators.get('aten::_scaled_dot_product_efficient_attention_backward', 0)
    math_calls = sum(n for key, n in operators.items()
                     if key.startswith('aten::') and 'scaled_dot_product_attention_math' in key)
    return {'operators': operators, 'efficient_forward_calls': forward,
            'efficient_backward_calls': backward, 'math_sdpa_calls': math_calls,
            'efficient_forward_and_backward_observed': forward > 0 and backward > 0,
            'no_math_sdpa_observed': math_calls == 0,
            'scope': 'CPU profiler observes dispatched ATen operators during the two real pilot steps; no tensor shapes/stacks retained.'}


def require_efficient_pilot(report, bypassed_calls):
    if bypassed_calls <= 0:
        raise RuntimeError('Corecția nu a fost traversată. Pilotul nu este confirmat.')
    if not report['efficient_forward_and_backward_observed'] or not report['no_math_sdpa_observed']:
        raise RuntimeError('Profilerul nu confirmă ruta memory-efficient fără SDPA math. Trimite raportul; nu porni antrenarea completă.')


In [ ]:
ATTENTION_FIX_SOURCE_SHA = '237106bbac04388e8a126a5e78331b5edc7f0e1b6d8756a4c9992633911aad12'
attention_fix = None
if not OFFLINE_VALIDATION and RUN_MODE != "prepare" and ENABLE_T4_KV_FIX:
    if MODEL_NAME != "unsloth/Qwen3.5-4B":
        raise RuntimeError("Această corecție este verificată doar pentru Qwen3.5-4B/T4; nu schimba modelul aici.")
    import torch, importlib, importlib.metadata as package_metadata
    sdpa_module = importlib.import_module("transformers.integrations.sdpa_attention")
    patch_versions = {name: package_metadata.version(name) for name in EXPECTED_VERSIONS}
    attention_fix = T4ExplicitKVFix(torch, sdpa_module, patch_versions)
    print("Corecție activă:", attention_fix.config())
else:
    print("Corecția nu este activată în prepare/offline sau este dezactivată explicit.")


Corecție activă: {'policy': 't4_explicit_kv_transformers_selector_v1', 'scope': 'T4 CUDA float16 K/V heads=4 head_dim=256 mask=None', 'versions': {'torch': '2.8.0', 'transformers': '5.2.0', 'unsloth': '2026.10.7', 'unsloth_zoo': '2026.10.7'}, 'transformers_sdpa_source_sha256': '87f933d1a2d8508df572da5c0748c6b24c22ff2b625796949957dcd86cc57564', 'model_weights_changed': False, 'dtype_changed': False, 'article_text_changed': False, 'selector': 'explicit_repeat_kv_via_existing_transformers_path'}


### 6. Încărcarea modelului și conversia strictă


In [ ]:
if not OFFLINE_VALIDATION and RUN_MODE != "prepare":
    if not CONTEXT_OK:
        raise RuntimeError("Context insuficient. Trimite raportul; nu rula antrenarea cu texte tăiate.")
    if RUN_MODE == "train" and not CONFIRM_FULL_TRAINING:
        raise RuntimeError("După pilot, setează CONFIRM_FULL_TRAINING=True explicit.")
    if "model" in globals():
        raise RuntimeError("Repornește sesiunea înainte de încărcarea repetată a modelului.")
    from unsloth import FastDecisionModel, DecisionTrainer, is_bfloat16_supported
    from transformers import set_seed
    import torch
    assert torch.cuda.is_available()
    set_seed(SEED)
    phase = "pilot_" + RUN_NAME if RUN_MODE == "pilot" else RUN_NAME
    RUN_DIR = SAVE_ROOT / phase
    if RUN_MODE == "train" and (RUN_DIR / "training_complete.json").exists():
        raise RuntimeError("Antrenarea este deja încheiată. Folosește calibrate/evaluate sau un RUN_NAME nou.")
    if RUN_MODE == "pilot" and (RUN_DIR / "pilot_complete.json").exists():
        raise RuntimeError("Pilotul este deja încheiat. Folosește un RUN_NAME nou pentru alt pilot.")
    if RUN_MODE == "train" and RESUME_CHECKPOINT is None and list((RUN_DIR / "checkpoints").glob("checkpoint-*")):
        raise RuntimeError("Există checkpointuri. Specifică RESUME_CHECKPOINT sau folosește un RUN_NAME nou.")
    load_path = str(RUN_DIR / "model_calibrated") if RUN_MODE == "evaluate" else str(RUN_DIR / "model_uncalibrated") if RUN_MODE == "calibrate" else MODEL_NAME
    if RUN_MODE == "evaluate" and not (RUN_DIR / "calibration_complete.json").exists():
        raise RuntimeError("Lipsește modelul calibrat al acestui experiment.")
    if RUN_MODE == "calibrate" and not (RUN_DIR / "training_complete.json").exists():
        raise RuntimeError("Antrenarea completă nu este salvată; reia checkpointul în modul train.")
    model, tokenizer = FastDecisionModel.from_pretrained(model_name=load_path,
        max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)
    if RUN_MODE in {"pilot", "train"}:
        model = FastDecisionModel.get_peft_model(model, r=16, lora_alpha=16,
            lora_dropout=0, use_gradient_checkpointing="unsloth", random_state=SEED)
    assert getattr(model, "is_clef", False), "API schimbat: acest notebook este verificat pentru Clef/Qwen3.5"
    items = runtime.strict_items(data, metadata, tokenizer, model, FastDecisionModel, length_report)
    run_config = {"model": MODEL_NAME, "max_seq_length": MAX_SEQ_LENGTH,
        "seed": SEED, "batch_size": TRAIN_BATCH_SIZE, "gradient_accumulation": 1 if RUN_MODE == "pilot" else GRADIENT_ACCUMULATION,
        "epochs": NUM_TRAIN_EPOCHS, "learning_rate": LEARNING_RATE,
        "lora_rank": 16, "bundle_manifest_sha256": EXPECTED_MANIFEST_SHA,
        "phase": "pilot" if RUN_MODE == "pilot" else "full", "no_truncation": True}
    import importlib.metadata as package_metadata, sys, inspect
    from unsloth.models import decision as decision_module
    versions = {}
    for package in ("unsloth", "unsloth_zoo", "torch", "transformers", "peft", "trl", "bitsandbytes"):
        try: versions[package] = package_metadata.version(package)
        except package_metadata.PackageNotFoundError: versions[package] = "unknown"
    run_config["runtime_versions"] = versions
    run_config["decision_source_sha256"] = runtime.sha_bytes(Path(inspect.getfile(decision_module)).read_bytes())
    run_config["base_revision"] = model.decision_config.get("base_revision")
    run_config["attention_fix"] = attention_fix.config() if attention_fix else {"policy": "original"}
    run_config["attention_fix_source_sha256"] = ATTENTION_FIX_SOURCE_SHA
    runtime.freeze_run(RUN_DIR, run_config)
    runtime.save_json(RUN_DIR / ("environment_" + RUN_MODE + ".json"), {
        "versions": versions, "python": sys.version, "gpu": torch.cuda.get_device_name(0),
        "decision_source_sha256": run_config["decision_source_sha256"],
        "decision_config": model.decision_config})
    print("Conversie integrală OK:", {k: len(v) for k, v in items.items()})
else:
    print("Încărcarea modelului și antrenarea sunt inactive în modul prepare/offline.")


==((====))==  Unsloth 2026.10.7: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Conversie integrală OK: {'train': 4448, 'calibration': 955, 'test': 954}


### 7. Pilot sau antrenare completă


In [ ]:
if not OFFLINE_VALIDATION and RUN_MODE in {"pilot", "train"}:
    from transformers import TrainingArguments
    import time
    if RUN_MODE == "pilot":
        selected = sorted(range(len(items["train"])), key=lambda i: len(items["train"][i]["input_ids"]))
        indexes = selected[-2:]
        training_items = [items["train"][i] for i in indexes]
    else:
        training_items = items["train"]
    if RESUME_CHECKPOINT is not None:
        checkpoint = Path(RESUME_CHECKPOINT).resolve()
        if checkpoint.parent != (RUN_DIR / "checkpoints").resolve() or not (checkpoint / "trainer_state.json").exists():
            raise RuntimeError("Checkpointul nu aparține acestui experiment sau nu conține starea trainerului.")
    arguments = TrainingArguments(output_dir=str(RUN_DIR / "checkpoints"),
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=1 if RUN_MODE == "pilot" else GRADIENT_ACCUMULATION,
        num_train_epochs=NUM_TRAIN_EPOCHS, max_steps=2 if RUN_MODE == "pilot" else -1,
        warmup_steps=0 if RUN_MODE == "pilot" else 10, learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine", weight_decay=.01,
        bf16=is_bfloat16_supported(), fp16=not is_bfloat16_supported(),
        eval_strategy="no", logging_steps=1, save_strategy="steps", save_steps=10,
        save_total_limit=2, save_only_model=False, report_to="none", seed=SEED)
    trainer = DecisionTrainer(model=model, processing_class=tokenizer,
        train_dataset=training_items, args=arguments)
    if attention_fix:
        attention_fix.assert_installed()
        attention_fix.reset_counters()
    if RUN_MODE == "pilot":
        assert TRAIN_BATCH_SIZE == 1 and len(training_items) == 2
        pilot_selection = [{"id": metadata["train"][i]["id"],
                            "tokens": len(items["train"][i]["input_ids"])} for i in indexes]
        runtime.save_json(RUN_DIR / "pilot_selection.json", pilot_selection)
        print("Articole pilot:", pilot_selection)
    torch.cuda.reset_peak_memory_stats()
    started = time.monotonic()
    from contextlib import nullcontext
    from torch.profiler import profile, ProfilerActivity
    profiling = profile(activities=[ProfilerActivity.CPU], record_shapes=False,
                        profile_memory=False, with_stack=False) if RUN_MODE == "pilot" else nullcontext()
    try:
        with profiling as attention_profile:
            training_stats = trainer.train(resume_from_checkpoint=RESUME_CHECKPOINT)
        torch.cuda.synchronize()
        if RUN_MODE == "pilot":
            backend_report = inspect_attention_events(attention_profile.key_averages())
            backend_report["attention_fix"] = attention_fix.snapshot() if attention_fix else None
            runtime.save_json(RUN_DIR / "attention_backend_report.json", backend_report)
            print(json.dumps(backend_report, indent=2))
            if attention_fix:
                require_efficient_pilot(backend_report, attention_fix.calls)
    except Exception as pilot_error:
        runtime.save_json(RUN_DIR / ("pilot_failed.json" if RUN_MODE == "pilot" else "training_failed.json"), {
            "exception_type": type(pilot_error).__name__, "message": str(pilot_error),
            "attention_fix": attention_fix.snapshot() if attention_fix else None,
            "peak_reserved_gib": torch.cuda.max_memory_reserved()/1024**3})
        raise
    training_seconds = time.monotonic()-started
    if not all(torch.isfinite(parameter).all().item() for parameter in model.parameters() if parameter.requires_grad):
        raise RuntimeError("Parametri antrenabili ne-finiți după pilot/antrenare; nu se marchează complet.")
    model.save_pretrained(str(RUN_DIR / "model_uncalibrated"))
    completed = {"attention_fix": attention_fix.snapshot() if attention_fix else None,
                 "mode": RUN_MODE, "training_metrics": training_stats.metrics,
                 "seconds": training_seconds,
                 "peak_reserved_gib": torch.cuda.max_memory_reserved()/1024**3}
    runtime.save_json(RUN_DIR / ("pilot_complete.json" if RUN_MODE == "pilot" else "training_complete.json"), completed)
    print(json.dumps(completed, indent=2))
    if RUN_MODE == "pilot":
        print("Pilot terminat. NU continua spre evaluarea finală. Repornește sesiunea înainte de train.")
else:
    print("Antrenarea nu este activată.")


Unsloth: Qwen3.5 overflows under float16 autocast, so Clef trains without autocast, keeping float32 activations.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,448 | Num Epochs = 1 | Total steps = 278
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 64,511,492 of 3,072,856,068 (2.10% trained)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
161,0.000662
162,0.000515


Step,Training Loss
161,0.000662
162,0.000515
163,0.000498
164,0.000490
165,0.000364
166,0.000555
167,0.000443
168,0.014172
169,0.000445
170,0.000362


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/SDCGIA/experiments/qwen35_4b_ro_t4_kv_v1/checkpoints/checkpoint-170/.unsloth-save-tmrr21xh/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/SDCGIA/experiments/qwen35_4b_ro_t4_kv_v1/checkpoints/checkpoint-180/.unsloth-save-2vcs_wi_/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/SDCGIA/experiments/qwen35_4b_ro_t4_kv_v1/checkpoints/checkpoint-190/.unsloth-save-h_lhje4i/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/SDCGIA/experiments/qwen35_4b_ro_t4_kv_v1/checkpoints/checkpoint-200/.unsloth-save-1kcoiqyt/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/SDCGIA/experiments/qwen35_4b_ro_t4_kv_v1/checkpoints/checkpoint-210/.unsloth-save-mx_902y7/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/

{
  "attention_fix": {
    "policy": "t4_explicit_kv_transformers_selector_v1",
    "scope": "T4 CUDA float16 K/V heads=4 head_dim=256 mask=None",
    "versions": {
      "torch": "2.8.0",
      "transformers": "5.2.0",
      "unsloth": "2026.10.7",
      "unsloth_zoo": "2026.10.7"
    },
    "transformers_sdpa_source_sha256": "87f933d1a2d8508df572da5c0748c6b24c22ff2b625796949957dcd86cc57564",
    "model_weights_changed": false,
    "dtype_changed": false,
    "article_text_changed": false,
    "selector": "explicit_repeat_kv_via_existing_transformers_path",
    "bypassed_native_gqa_calls": 30208,
    "observed_key_shapes_before_repeat": [
      [
        1,
        4,
        277,
        256
      ],
      [
        1,
        4,
        354,
        256
      ],
      [
        1,
        4,
        356,
        256
      ],
      [
        1,
        4,
        363,
        256
      ],
      [
        1,
        4,
        389,
        256
      ],
      [
        1,
        4,
  

### 8. Calibrare și salvarea modelului final


In [ ]:
if not OFFLINE_VALIDATION and RUN_MODE in {"train", "calibrate"}:
    if not (RUN_DIR / "training_complete.json").exists():
        raise RuntimeError("Antrenarea completă nu este confirmată.")
    calibration_metrics = FastDecisionModel.calibrate(model, tokenizer, items["calibration"], batch_size=1)
    runtime.save_json(RUN_DIR / "calibration_metrics.json", calibration_metrics)
    model.save_pretrained(str(RUN_DIR / "model_calibrated"))
    saved_files = list((RUN_DIR / "model_calibrated").iterdir())
    assert any(p.name.startswith("adapter_model") for p in saved_files)
    assert (RUN_DIR / "model_calibrated" / "joint_head.safetensors").exists(), "Decision head lipsă"
    final_fingerprint = runtime.adapter_fingerprint(RUN_DIR / "model_calibrated")
    runtime.save_json(RUN_DIR / "calibration_complete.json", {"adapter_fingerprint": final_fingerprint})
    print("Model calibrat salvat în:", RUN_DIR / "model_calibrated")
elif not OFFLINE_VALIDATION and RUN_MODE == "evaluate":
    final_fingerprint = runtime.adapter_fingerprint(RUN_DIR / "model_calibrated")
    saved = json.loads((RUN_DIR / "calibration_complete.json").read_text())
    assert saved["adapter_fingerprint"] == final_fingerprint
    print("Model calibrat reîncărcat; configurația și adapterul sunt verificate.")
else:
    print("Calibrarea nu este activată.")


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/SDCGIA/experiments/qwen35_4b_ro_t4_kv_v1/model_calibrated/.unsloth-save-5agji5dj/tokenizer_config.json.


Model calibrat salvat în: /content/drive/MyDrive/SDCGIA/experiments/qwen35_4b_ro_t4_kv_v1/model_calibrated


In [ ]:
import json
from pathlib import Path

run = Path(
    "/content/drive/MyDrive/SDCGIA/experiments/"
    "qwen35_4b_ro_t4_kv_v1"
)

for name in [
    "calibration_metrics.json",
    "calibration_complete.json",
    "model_calibrated/adapter_model.safetensors",
    "model_calibrated/joint_head.safetensors",
]:
    path = run / name
    assert path.is_file() and path.stat().st_size > 0, f"Lipsă/gol: {name}"
    print("OK:", name)

print(json.dumps(
    json.loads((run / "calibration_metrics.json").read_text()),
    ensure_ascii=False,
    indent=2,
))

OK: calibration_metrics.json
OK: calibration_complete.json
OK: model_calibrated/adapter_model.safetensors
OK: model_calibrated/joint_head.safetensors
{
  "accuracy": 1.0,
  "ece": 2.6213561321597467e-09,
  "loss": 2.6213559341796617e-09,
  "record_accuracy": 1.0,
  "fitted_types": [
    0
  ]
}


## Checks


In [ ]:
RUN_FINAL_TEST = True  # Schimbă în True numai după antrenare și calibrare.
if not OFFLINE_VALIDATION and RUN_FINAL_TEST:
    if RUN_MODE not in {"train", "calibrate", "evaluate"} or not (RUN_DIR / "calibration_complete.json").exists():
        raise RuntimeError("Este necesar modelul final calibrat, nu pilotul.")
    FastDecisionModel.for_inference(model)
    predictions = runtime.predict_saved(data["test"], metadata["test"],
        lambda state, questions: FastDecisionModel.predict(model, tokenizer, state, questions),
        RUN_DIR / "test_predictions.jsonl", final_fingerprint + EXPECTED_MANIFEST_SHA)
    test_results = runtime.grouped_metrics(predictions)
    runtime.save_json(RUN_DIR / "test_metrics.json", test_results)
    print(json.dumps(test_results, ensure_ascii=False, indent=2))
else:
    print("Testul final nu este activat. Modelul salvat și checkpointurile nu sunt pierdute.")


Predicții salvate: 25/954
Predicții salvate: 50/954
Predicții salvate: 75/954
Predicții salvate: 100/954
Predicții salvate: 125/954
Predicții salvate: 150/954
Predicții salvate: 175/954
Predicții salvate: 200/954
Predicții salvate: 225/954
Predicții salvate: 250/954
Predicții salvate: 275/954
Predicții salvate: 300/954
Predicții salvate: 325/954
Predicții salvate: 350/954
Predicții salvate: 375/954
Predicții salvate: 400/954
Predicții salvate: 425/954
Predicții salvate: 450/954
Predicții salvate: 475/954
Predicții salvate: 500/954
Predicții salvate: 525/954
Predicții salvate: 550/954
Predicții salvate: 575/954
Predicții salvate: 600/954
Predicții salvate: 625/954
Predicții salvate: 650/954
Predicții salvate: 675/954
Predicții salvate: 700/954
Predicții salvate: 725/954
Predicții salvate: 750/954
Predicții salvate: 775/954
Predicții salvate: 800/954
Predicții salvate: 825/954
Predicții salvate: 850/954
Predicții salvate: 875/954
Predicții salvate: 900/954
Predicții salvate: 925/954
Pred

### 10. Arhivă de siguranță


In [ ]:
if not OFFLINE_VALIDATION and RUN_MODE in {"train", "calibrate", "evaluate"}:
    if not (RUN_DIR / "calibration_complete.json").exists():
        raise RuntimeError("Modelul final nu este încă salvat.")
    import zipfile, hashlib
    archive_path = RUN_DIR / "human_ai_final.zip"
    temporary = archive_path.with_suffix(".zip.tmp")
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(RUN_DIR.rglob("*")):
            relative = path.relative_to(RUN_DIR)
            if path.is_file() and "checkpoints" not in relative.parts and path not in {archive_path, temporary}:
                archive.write(path, relative.as_posix())
    with zipfile.ZipFile(temporary) as archive:
        assert archive.testzip() is None
        for path in (RUN_DIR / "model_calibrated").rglob("*"):
            if path.is_file():
                assert archive.read(path.relative_to(RUN_DIR).as_posix()) == path.read_bytes()
    temporary.replace(archive_path)
    print("Salvare verificată:", archive_path)
    print("Dimensiune MB:", round(archive_path.stat().st_size/1024**2, 2))
else:
    print("Nu există încă un model final de arhivat.")


Salvare verificată: /content/drive/MyDrive/SDCGIA/experiments/qwen35_4b_ro_t4_kv_v1/human_ai_final.zip
Dimensiune MB: 332.14


In [ ]:
# Încărcare probă pentru experimentul suplimentar
from google.colab import files
supp_probe_upload = files.upload()
print("Fișiere primite:", list(supp_probe_upload))

Saving supplementary_evaluation.zip to supplementary_evaluation.zip
Fișiere primite: ['supplementary_evaluation.zip']


In [24]:
from pathlib import Path
SUPPLEMENT_BUNDLE = Path("/content/supplementary_evaluation.zip")
# Experiment suplimentar separat — fără antrenare și fără calibrare
from pathlib import Path
import types
from unsloth.models.clef import encode_record
supplement_module = types.ModuleType('sdcgia_supplementary_evaluation')
exec(compile('"""Evaluate the supplementary bundle only; no fit/calibrate/save_pretrained calls."""\nfrom collections import Counter\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport zipfile\n\nEXPECTED_MODEL = \'bde45980a57d5a2be449e6442f899146ed6eff87d192490b85410330793bd140\'\nQUESTIONS = {\'origin\': {\'type\':\'choice\',\n    \'instructions\':\'Stabilește originea textului: redactat de un om sau generat de inteligență artificială.\',\n    \'criteria\':{\'human\':\'Text redactat de un om.\',\'ai\':\'Text generat de inteligență artificială.\'}}}\n\n\ndef sha(data):\n    return hashlib.sha256(data).hexdigest()\n\n\ndef load_bundle(path):\n    with zipfile.ZipFile(path) as z:\n        assert set(z.namelist())=={\'manifest.json\',\'protocol.json\',\'evaluation.jsonl\'}\n        assert z.testzip() is None\n        manifest=json.loads(z.read(\'manifest.json\'))\n        for name,digest in manifest[\'files\'].items():\n            assert sha(z.read(name))==digest\n        protocol=json.loads(z.read(\'protocol.json\'))\n        rows=[json.loads(s) for s in z.read(\'evaluation.jsonl\').decode().splitlines()]\n    assert manifest[\'model_fingerprint\']==EXPECTED_MODEL\n    assert protocol[\'no_training\'] is True and protocol[\'no_calibration\'] is True\n    assert protocol[\'threshold\']==.5\n    assert len(rows)==manifest[\'rows\'] and len({r[\'id\'] for r in rows})==len(rows)\n    assert sum(r[\'label\']==\'human\' for r in rows)==30\n    assert all(r[\'text\'].strip() and sha(r[\'text\'].encode())==r[\'text_sha256\'] for r in rows)\n    assert all(r[\'label\'] in {\'human\',\'ai\'} for r in rows)\n    assert manifest[\'human\']==30 and manifest[\'ai_nonempty\']==len(rows)-30\n    source_ids={r[\'source_id\'] for r in rows if r[\'label\']==\'human\'}\n    assert len(source_ids)==30\n    ai=[r for r in rows if r[\'label\']==\'ai\']\n    assert len({(r[\'source_id\'],r[\'model\'],r[\'profile\']) for r in ai})==len(ai)\n    assert all(r[\'source_id\'] in source_ids and r[\'model\'] in protocol[\'models\']\n        and r[\'profile\'] in protocol[\'profiles\'] for r in ai)\n    return rows,protocol,manifest\n\n\ndef evaluate(bundle,output,model_dir,model,tokenizer,fast_model,runtime,encode_record):\n    output,model_dir=Path(output),Path(model_dir)\n    # Output is separate from the immutable original experiment and model files.\n    assert output.resolve()!=model_dir.parent.resolve()\n    assert model_dir.resolve() not in output.resolve().parents\n    fingerprint=runtime.adapter_fingerprint(model_dir)\n    assert fingerprint==EXPECTED_MODEL,\'The calibrated model files do not match the frozen model\'\n    assert json.loads((model_dir.parent/\'calibration_complete.json\').read_text())[\'adapter_fingerprint\']==fingerprint\n    inputs,protocol,manifest=load_bundle(bundle)\n    assert int(model.decision_config[\'max_len\'])==12288\n    output.mkdir(parents=True,exist_ok=True)\n    identity={\'model_fingerprint\':fingerprint,\'bundle_sha256\':sha(Path(bundle).read_bytes()),\n        \'threshold\':.5,\'ids\':[r[\'id\'] for r in inputs]}\n    identity_path=output/\'prediction_identity.json\'\n    if identity_path.exists():\n        assert json.loads(identity_path.read_text())==identity,\'Prediction cache is for different inputs/model\'\n    else:\n        runtime.save_json(identity_path,identity)\n    data=[{\'state\':r[\'text\'],\'questions\':QUESTIONS,\'gold\':{\'origin\':r[\'label\']}} for r in inputs]\n    metadata=[{k:v for k,v in r.items() if k!=\'text\'} for r in inputs]\n    # from_pretrained may return a multimodal processor. encode_record needs\n    # its text tokenizer; predict still receives the original processor.\n    text_tokenizer=getattr(tokenizer,\'tokenizer\',tokenizer)\n    report=runtime.inspect_lengths({\'supplement\':data},{\'supplement\':metadata},\n        lambda row,budget:encode_record(text_tokenizer,row,max_length=budget),12288)\n    assert report[\'over_context\']==0,\'No truncation is permitted\'\n    runtime.save_json(output/\'token_lengths.json\',report)\n    fast_model.for_inference(model)\n    predictions=runtime.predict_saved(data,metadata,\n        lambda state,questions:fast_model.predict(model,tokenizer,state,questions),\n        output/\'predictions.jsonl\',fingerprint+identity[\'bundle_sha256\'])\n    assert len(predictions)==len(inputs)\n    human=[r for r in predictions if r[\'label\']==\'human\']\n    metrics={\'experiment_id\':manifest[\'experiment_id\'],\'threshold\':.5,\n        \'overall\':runtime.metrics(predictions),\n        \'unique_human_controls\':len(human),\'attempted_ai\':120,\'included_nonempty_ai\':len(predictions)-30,\n        \'per_cell\':{},\'sampling_paired_tests\':{},\n        \'probabilities_at_extremes\':sum(r[\'p_ai\'] in (0.,1.) for r in predictions),\n        \'caveats\':protocol[\'source_limitations\']+[\n            \'Global accuracy uses 120 planned AI texts and only 30 unique human controls, unlike the original test balance.\',\n            \'The same 30 human controls are reused for every cell; do not sum subgroup denominators.\',\n            \'Paired tests are descriptive, unadjusted for multiple comparisons; small samples cannot establish a generator ranking.\']}\n    for model_name in protocol[\'models\']:\n        for profile in protocol[\'profiles\']:\n            ai=[r for r in predictions if r[\'model\']==model_name and r[\'profile\']==profile]\n            if ai:\n                metrics[\'per_cell\'][model_name+\' / \'+profile]=runtime.metrics(human+ai)\n        maps={profile:{r[\'source_id\']:r for r in predictions if r[\'model\']==model_name and r[\'profile\']==profile}\n            for profile in protocol[\'profiles\']}\n        left,right=maps[\'conservative\'],maps[\'varied\']\n        common=set(left)&set(right)\n        b=sum(left[s][\'predicted_label\']==\'ai\' and right[s][\'predicted_label\']!=\'ai\' for s in common)\n        c=sum(left[s][\'predicted_label\']!=\'ai\' and right[s][\'predicted_label\']==\'ai\' for s in common)\n        n=b+c\n        p_value=min(1.,2*sum(math.comb(n,i) for i in range(min(b,c)+1))/(2**n)) if n else 1.\n        metrics[\'sampling_paired_tests\'][model_name]={\'paired_sources\':len(common),\n            \'conservative_only_detected\':b,\'varied_only_detected\':c,\'mcnemar_exact_two_sided_p\':p_value}\n    runtime.save_json(output/\'metrics.json\',metrics)\n    runtime.save_json(output/\'evaluation_complete.json\',identity)\n    assert runtime.adapter_fingerprint(model_dir)==fingerprint,\'Calibrated model files changed\'\n    archive=output/\'supplementary_results.zip\'\n    with zipfile.ZipFile(archive,\'w\',compression=zipfile.ZIP_DEFLATED) as z:\n        for path in sorted(output.iterdir()):\n            if path.is_file() and path.suffix in {\'.json\',\'.jsonl\'}:\n                z.write(path,path.name)\n    with zipfile.ZipFile(archive) as z:\n        assert z.testzip() is None\n    print(json.dumps(metrics,ensure_ascii=False,indent=2))\n    print(\'Rezultate suplimentare salvate:\',archive)\n    return archive\n', 'supplementary_sampling_evaluation.py', 'exec'), supplement_module.__dict__)
SUPPLEMENT_RESULT = supplement_module.evaluate(
    SUPPLEMENT_BUNDLE,
    Path('/content/drive/MyDrive/SDCGIA/experiments/qwen25_sampling_supplement_v1'),
    Path('/content/drive/MyDrive/SDCGIA/experiments/qwen35_4b_ro_t4_kv_v1/model_calibrated'),
    model, tokenizer, FastDecisionModel, runtime, encode_record)


Predicții salvate: 25/150
Predicții salvate: 50/150
Predicții salvate: 75/150
Predicții salvate: 100/150
Predicții salvate: 125/150
Predicții salvate: 150/150
{
  "experiment_id": "qwen25_sampling_supplement_v1",
  "threshold": 0.5,
  "overall": {
    "n": 150,
    "human_n": 30,
    "ai_n": 120,
    "tp": 120,
    "tn": 30,
    "fp": 0,
    "fn": 0,
    "accuracy": 1.0,
    "false_positive_rate": 0.0,
    "false_negative_rate": 0.0,
    "ai_recall": 1.0,
    "fpr_wilson95": [
      0,
      0.11351339317396321
    ],
    "ai_recall_wilson95": [
      0.9689808335812982,
      1
    ],
    "balanced_accuracy": 1.0,
    "brier": 0.0,
    "log_loss": -0.0,
    "ece_10_equal_width_confidence_bins": 0.0
  },
  "unique_human_controls": 30,
  "attempted_ai": 120,
  "included_nonempty_ai": 120,
  "per_cell": {
    "qwen2.5:1.5b / conservative": {
      "n": 60,
      "human_n": 30,
      "ai_n": 30,
      "tp": 30,
      "tn": 30,
      "fp": 0,
      "fn": 0,
      "accuracy": 1.0,
      "fa

## Next Steps
